In [12]:
import numpy as np

In [13]:
weights = np.array([
    [-1.8,-0.9,0.0,0.7,1.5],
    [-2.4,-0.3,0.2,1.1,2.0]
],dtype=np.float32)

activations=np.array([
    [0.0,0.3,0.8,1.4,2.1],
    [0.1,0.6,1.0,1.8,3.2]
],dtype=np.float32)

In [14]:
def symmetric_quantize(tensor):

    x_abs=np.max(np.abs(tensor))

    scale=x_abs/127

    zero_point=0

    q=np.round(tensor/scale)

    q=np.clip(q,-127,127)

    return q.astype(np.int8),scale,zero_point

In [15]:
def asymmetric_quantize(tensor):

    x_min=np.min(tensor)

    x_max=np.max(tensor)

    scale=(x_max-x_min)/255

    if scale==0:
        scale=1.0

    zero_point=round(-128-x_min/scale)

    zero_point=int(np.clip(zero_point,-128,127))

    q=np.round(tensor/scale)+zero_point

    q=np.clip(q,-128,127)

    return q.astype(np.int8),scale,zero_point

In [16]:
def dequantize(q,scale,zero_point):

    return (q.astype(np.float32)-zero_point)*scale

In [17]:
def calculate_metrics(original,dequantized,q):

    error=np.abs(original-dequantized)

    mae=np.mean(error)

    mse=np.mean((original-dequantized)**2)

    max_error=np.max(error)

    sat_min=np.sum(q==-128)

    sat_max=np.sum(q==127)

    sat_total=sat_min+sat_max

    return mae,mse,max_error,sat_min,sat_max,sat_total

In [18]:
comparison = []

for name, tensor in [("Weights", weights),
                     ("Activations", activations)]:

    # Symmetric
    q_sym, scale_sym, zp_sym = symmetric_quantize(tensor)

    dq_sym = dequantize(q_sym, scale_sym, zp_sym)

    mae_sym, mse_sym, max_sym, satmin_sym, satmax_sym, sattotal_sym = \
        calculate_metrics(tensor, dq_sym, q_sym)

    # Asymmetric
    q_asym, scale_asym, zp_asym = asymmetric_quantize(tensor)

    dq_asym = dequantize(q_asym, scale_asym, zp_asym)

    mae_asym, mse_asym, max_asym, satmin_asym, satmax_asym, sattotal_asym = \
        calculate_metrics(tensor, dq_asym, q_asym)

    comparison.append([
        name,
        "Symmetric",
        scale_sym,
        zp_sym,
        mae_sym,
        mse_sym,
        max_sym,
        satmin_sym,
        satmax_sym,
        sattotal_sym
    ])

    comparison.append([
        name,
        "Asymmetric",
        scale_asym,
        zp_asym,
        mae_asym,
        mse_asym,
        max_asym,
        satmin_asym,
        satmax_asym,
        sattotal_asym
    ])

    print("="*70)
    print(name)
    print("="*70)

    print("\nOriginal Tensor")
    print(tensor)

    print("\nSymmetric Quantized")
    print(q_sym)

    print("\nSymmetric Dequantized")
    print(dq_sym)

    print("\nAsymmetric Quantized")
    print(q_asym)

    print("\nAsymmetric Dequantized")
    print(dq_asym)

Weights

Original Tensor
[[-1.8 -0.9  0.   0.7  1.5]
 [-2.4 -0.3  0.2  1.1  2. ]]

Symmetric Quantized
[[ -95  -48    0   37   79]
 [-127  -16   11   58  106]]

Symmetric Dequantized
[[-1.7952756  -0.9070866   0.          0.6992126   1.4929134 ]
 [-2.4        -0.3023622   0.20787401  1.096063    2.0031495 ]]

Asymmetric Quantized
[[ -93  -41   11   52   98]
 [-128   -6   23   75  127]]

Asymmetric Dequantized
[[-1.7945098  -0.8972549   0.          0.707451    1.5011765 ]
 [-2.3984313  -0.29333332  0.20705882  1.1043137   2.0015686 ]]
Activations

Original Tensor
[[0.  0.3 0.8 1.4 2.1]
 [0.1 0.6 1.  1.8 3.2]]

Symmetric Quantized
[[  0  12  32  56  83]
 [  4  24  40  71 127]]

Symmetric Dequantized
[[0.        0.3023622 0.8062992 1.4110236 2.0913386]
 [0.1007874 0.6047244 1.007874  1.7889764 3.2      ]]

Asymmetric Quantized
[[-128 -104  -64  -16   39]
 [-120  -80  -48   15  127]]

Asymmetric Dequantized
[[0.         0.30117646 0.80313724 1.4054902  2.0956862 ]
 [0.10039216 0.6023529  1

In [19]:
print("="*120)

print("{:<15}{:<15}{:<12}{:<12}{:<12}{:<12}{:<12}{:<10}{:<10}{:<10}".format(
    "Tensor",
    "Method",
    "Scale",
    "ZeroPt",
    "MAE",
    "MSE",
    "MaxErr",
    "SatMin",
    "SatMax",
    "SatTotal"
))

print("="*120)

for row in comparison:

    print("{:<15}{:<15}{:<12.6f}{:<12}{:<12.6f}{:<12.6f}{:<12.6f}{:<10}{:<10}{:<10}".format(
        row[0],
        row[1],
        row[2],
        row[3],
        row[4],
        row[5],
        row[6],
        row[7],
        row[8],
        row[9]
    ))

Tensor         Method         Scale       ZeroPt      MAE         MSE         MaxErr      SatMin    SatMax    SatTotal  
Weights        Symmetric      0.018898    0           0.003701    0.000022    0.007874    0         0         0         
Weights        Asymmetric     0.017255    11          0.003804    0.000021    0.007451    1         1         2         
Activations    Symmetric      0.025197    0           0.005276    0.000045    0.011024    0         1         1         
Activations    Asymmetric     0.012549    -128        0.002627    0.000011    0.005490    1         1         2         


In [20]:
outlier_tensor = np.array(
    [-0.5, -0.2, 0.0, 0.3, 0.7, 12.0],
    dtype=np.float32
)

without_outlier = np.array(
    [-0.5, -0.2, 0.0, 0.3, 0.7],
    dtype=np.float32
)

In [21]:
def evaluate_tensor(name, tensor):

    print("\n")
    print("="*70)
    print(name)
    print("="*70)

    # Symmetric
    q, scale, zp = symmetric_quantize(tensor)

    dq = dequantize(q, scale, zp)

    error = np.abs(tensor - dq)

    mae, mse, max_error, *_ = calculate_metrics(tensor, dq, q)

    print("\nSymmetric")

    print("Scale :", scale)

    print("Zero Point :", zp)

    print("Quantized :", q)

    print("Dequantized :", dq)

    print("Per-element Error :", error)

    print("MAE :", mae)

    print("MSE :", mse)

    print("Max Error :", max_error)

    # Asymmetric
    q, scale, zp = asymmetric_quantize(tensor)

    dq = dequantize(q, scale, zp)

    error = np.abs(tensor - dq)

    mae, mse, max_error, *_ = calculate_metrics(tensor, dq, q)

    print("\nAsymmetric")

    print("Scale :", scale)

    print("Zero Point :", zp)

    print("Quantized :", q)

    print("Dequantized :", dq)

    print("Per-element Error :", error)

    print("MAE :", mae)

    print("MSE :", mse)

    print("Max Error :", max_error)

In [22]:
evaluate_tensor("With Outlier", outlier_tensor)

evaluate_tensor("Without Outlier", without_outlier)



With Outlier

Symmetric
Scale : 0.09448819
Zero Point : 0
Quantized : [ -5  -2   0   3   7 127]
Dequantized : [-0.47244096 -0.18897638  0.          0.28346455  0.6614173  12.        ]
Per-element Error : [0.02755904 0.01102363 0.         0.01653546 0.03858268 0.        ]
MAE : 0.015616802
MSE : 0.000440511
Max Error : 0.038582683

Asymmetric
Scale : 0.04901961
Zero Point : -118
Quantized : [-128 -122 -118 -112 -104  127]
Dequantized : [-0.49019608 -0.19607843  0.          0.29411766  0.6862745  12.009804  ]
Per-element Error : [0.00980392 0.00392157 0.         0.00588235 0.01372546 0.00980377]
MAE : 0.0071895123
MSE : 7.176663e-05
Max Error : 0.01372546


Without Outlier

Symmetric
Scale : 0.005511811
Zero Point : 0
Quantized : [-91 -36   0  54 127]
Dequantized : [-0.5015748 -0.1984252  0.         0.2976378  0.7      ]
Per-element Error : [0.00157481 0.0015748  0.         0.00236222 0.        ]
MAE : 0.0011023671
MSE : 2.1080245e-06
Max Error : 0.0023622215

Asymmetric
Scale : 0.0047